## WEEK - 8 Advanced RAG & Hybrid Retrieval
## WEEK - 10 LLaMA Fine-Tuning with LoRA

In [ ]:
# RAG using langchain

In [ ]:
from langchain.agents import create_agent

In [ ]:
# create model
import os
from langchain.chat_models import init_chat_model

# os.environ["HUGGINGFACEHUB_API_TOKEN"] = ""

model = init_chat_model(
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    model_provider="huggingface",
    temperature=0.7,
    max_tokens=1024,
)

In [ ]:
# define embedding
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

In [ ]:
# define vector store
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore(embeddings)

In [ ]:
# extract content from website
import bs4
from langchain_community.document_loaders import WebBaseLoader

bs4_strainer = bs4.SoupStrainer(class_=("post-title", "post-header", "post-content"))
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs={"parse_only": bs4_strainer},
)
docs = loader.load()

assert len(docs) == 1
print(f"Total characters: {len(docs[0].page_content)}")

In [ ]:
# divide docments into chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200, add_start_index=True
)
split_docs = splitter.split_documents(docs)

print(f"This blog split into {len(split_docs)} sub documents")

In [ ]:
# make each doc id for fast searching
doc_id = vector_store.add_documents(documents=split_docs)

In [ ]:
@dynamic_prompt
def prompt_with_context(request: ModelRequest) -> str:
    last_query = request.state["messages"][-1].text

    retrieved_docs = vector_store.similarity_search_with_score(last_query, k=3)

    filtered_docs = [doc for doc, score in retrieved_docs if score < 0.5]

    if not filtered_docs:
        return (
            "You MUST respond with EXACTLY:\n"
            "'I don't know based on the given context.'"
        )

    docs_content = "\n\n".join(doc.page_content for doc in filtered_docs)

    system_message = (
        "You MUST answer ONLY from the provided context.\n"
        "If the answer is not explicitly present, say EXACTLY:\n"
        "'I don't know based on the given context.'\n"
        "Do NOT use prior knowledge.\n"
        "Do NOT guess.\n\n"
        f"Context:\n{docs_content}"
    )

    return system_message


agent = create_agent(model, tools=[], middleware=[prompt_with_context])
query = "What is neuro science"
for step in agent.stream(
    {"messages": [{"role": "user", "content": query}]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

In [ ]:
RAG with langchain, open-closed model and model serving

Learned how to implement RAG with langchain framework and also implement and unerstand difference between open-source and closed model.learned model serving and framework that are use and difference between model serving and model deployment.


In [ ]:
# 03-04-26

In [ ]:
%pip install PyMuPDF

In [ ]:
import fitz

In [ ]:
# Extract text with page number
def load_file(file_path):
    file = fitz.open(file_path)
    pages = []

    for i, page in enumerate(file):
        text = page.get_text()
        pages.append({"page": i + 1, "text": text})
    return pages

In [ ]:
def parse_pdf(pages):
    parsed_docs = []

    for page_data in pages:
        page_num = page_data["page"]
        lines = [l.strip() for l in page_data["text"].split("\n") if l.strip()]

        parent_topic = None
        description = ""

        for line in lines:

            # Detect parent topic
            if line.lower().startswith("topic:"):
                parent_topic = line.replace("TOPIC:", "").strip()

            # Detect description
            elif line.lower().startswith("description:"):
                description = line.replace("DESCRIPTION:", "").strip()

            # Detect subtopic
            elif line.lower().startswith("sub topic:"):
                sub_line = line.replace("sub topic:", "").strip()

                if "–" in sub_line:
                    sub_title, sub_desc = sub_line.split("–", 1)
                else:
                    sub_title, sub_desc = sub_line, ""

                parsed_docs.append(
                    {
                        "content": sub_desc.strip(),
                        "metadata": {
                            "page": page_num,
                            "parent_topic": parent_topic,
                            "description": description,
                            "sub_topic": sub_title.strip(),
                        },
                    }
                )

    return parsed_docs

In [ ]:
# convert to langchain document
from langchain_core.documents import Document


def create_documents(parsed_docs):
    return [
        Document(page_content=d["content"], metadata=d["metadata"]) for d in parsed_docs
    ]

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


def chunking(document):
    splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
    chunked_doc = splitter.split_documents(document)
    return chunked_doc

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS


def create_vectorstore(document):
    embeddings = HuggingFaceEmbeddings(model="all-MiniLM-L6-V2")
    return FAISS.from_documents(document, embeddings)

In [ ]:
def query(vector_store, query):
    results = vector_store.similarity_search(query, k=3)

    final_output = ""

    for doc in results:
        final_output += f"""
Answer:{doc.page_content}
(Page:{doc.metadata['page']} | Topic : {doc.metadata['parent_topic']} | subtopic : {doc.metadata['sub_topic']})
"""
    return final_output

In [ ]:
file_path = "/home/shrutik/Documents/doc.pdf"

# 1. Load PDF
pages = load_file(file_path)

# 2. Parse structured data
parsed_docs = parse_pdf(pages)

# 3. Convert to documents
documents = create_documents(parsed_docs)

chunked = chunking(documents)
# 4. Create vector DB
vector_store = create_vectorstore(chunked)

In [ ]:
# 5. Query
print(query(vector_store, "what is NLP?"))

Cost optimization,RAG(metadata) & Quantization

Today i implement RAG with langchain extracting metadata like page number,topic and subtopic of the particular topic.Learned cost optimization concept and quantization(16-bit,8-bit,4-bit) concept and basics of LoRA(Low rank adaptation)

In [ ]:
# 06-05-26

## Method of Quantization
- Post-training quantization
- Quantization-Aware Training 

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

In [ ]:
# Post-training quantization

In [ ]:
# create basic model
model = keras.Sequential(
    [
        layers.Dense(64, activation="relu", input_shape=(10,)),
        layers.Dense(32, activation="relu"),
        layers.Dense(1),
    ]
)

model.compile(optimizer="adam", loss="mse")

import numpy as np

x = np.random.rand(100, 10)
y = np.random.rand(100, 1)

model.fit(x, y, epochs=3)
model.save("new_model.h5")

In [ ]:
# Post-training quantization (Tensorflow)

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
# enable quantization
converter.optimizations = [tf.lite.Optimize.DEFAULT]
quantized_model = converter.convert()
with open("quant_tflite_model", "wb") as f:
    f.write(quantized_model)

In [ ]:
# Quantization-Aware Training (Tensorflow)

In [ ]:
import tensorflow_model_optimization as tfmot

quantize_model = tfmot.quantization.keras.quantize_model

qat_model = quantize_model(model)
qat_model.compile(optimizer="adam", loss="mse", metrics=["mae"])
qat_model.fit(x, y, epochs=5)
converter = tf.lite.TFLiteConverter.from_keras_model(qat_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()
with open("qat_model.tflite", "wb") as f:
    f.write(tflite_model)

In [ ]:
# Quantization using PyTorch

In [ ]:
# dynamic quantization


In [ ]:
torch.save(model_pt, "w+")

In [ ]:
import torch
import torch.nn as nn

from torchao.quantization import quantize_
from torchao.quantization import Int8DynamicActivationInt8WeightConfig


model_pt = nn.Sequential(
    nn.Linear(10, 40), nn.ReLU(), nn.Linear(40, 30), nn.ReLU(), nn.Linear(30, 1)
)

model_pt.eval()

# ✅ Create dynamic quantization config
quant_config = Int8DynamicActivationInt8WeightConfig()

# ✅ Apply quantization (IN-PLACE)
quantize_(model_pt, quant_config)

# ✅ Test
data = torch.randn(10, 10)
output = model_pt(data)

print(output)


In [ ]:
# static quantization

In [ ]:
import torch.quantization as quant

model_pt.eval()
model_pt.qconfig = quant.get_default_qconfig("fbgemm")
torch.quant.prepare(model_pt, inplace=True)

for i in range(100):
    data = torch.randn(1, 10)
    model_pt(data)

torch.quant.convert(model_pt, inplace=True)

In [ ]:
# QAT

In [ ]:
import torch
import torch.nn as nn
import torch.ao.quantization as quant

# ✅ 1. Set backend (IMPORTANT)
torch.backends.quantized.engine = "fbgemm"


# ✅ 2. Define model with Quant + DeQuant
class QATModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.quant = quant.QuantStub()
        self.fc1 = nn.Linear(10, 20)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(20, 1)
        self.dequant = quant.DeQuantStub()

    def forward(self, x):
        x = self.quant(x)  # float → fake int8
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.dequant(x)  # back to float
        return x


# ✅ 3. Create model
model = QATModel()

# ✅ 4. Set QAT config
model.qconfig = quant.get_default_qat_qconfig("fbgemm")

# ✅ 5. Prepare for QAT (adds fake quant)
torch.quant.prepare_qat(model_pt, inplace=True)

# ✅ 6. Training setup
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.MSELoss()

model.train()

# ✅ 7. Dummy training loop
for epoch in range(5):
    data = torch.randn(10, 10)
    target = torch.randn(10, 1)

    optimizer.zero_grad()
    output = model(data)
    loss = loss_fn(output, target)
    loss.backward()
    optimizer.step()

    print(f"Epoch {epoch+1}, Loss: {loss.item()}")

# ✅ 8. Convert to real quantized model
model.eval()
torch.quant.convert(model_pt, inplace=True)

# ✅ 9. Test quantized model
data = torch.randn(10, 10)
output = model(data)

print("\nQuantized model output:")
print(output)

## Perform fine-tuning using LoRA

In [ ]:
from typing import Dict, List
from datasets import Dataset, load_dataset, disable_caching
disable_caching() ## disable huggingface cache
import torch
from torch.utils.data import Dataset
from IPython.display import Markdown
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer
from transformers import AutoTokenizer, DataCollatorForLanguageModeling
# dataset creation
dataset = load_dataset("MBZUAI/LaMini-instruction" , split = 'train')
small_dataset = dataset.select(i for i in range(200))
print(small_dataset[0])
# creating template 
prompt_template = """ Below is an instruction that describes a task. Write a response that appropriately completes the request. Instruction: {instruction}\n Response:"""
answer_template = """{response}"""
def _add_text(rec):
    instruction = rec["instruction"]
    response = rec["response"]

    # check if both exist
    if not instruction:
        raise ValueError(f"Expected a instruction in: {rec}")
    if not response:
        raise ValueError(f"Expected a response in: {rec}")
    rec["prompt"] = prompt_template.format(instruction=instruction)
    rec["answer"] = answer_template.format(response=response)
    rec["text"] = rec["prompt"] + rec["answer"]
    return rec

small_dataset = small_dataset.map(_add_text)
print(small_dataset[0])
import torch
model_id = "Qwen/Qwen3-0.6B"
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype = torch.float32
)
model.resize_token_embeddings(len(tokenizer))
from functools import partial
import copy
from transformers import DataCollatorForSeq2Seq

MAX_LENGTH = 256

# Function to generate token embeddings from text part of batch
def _preprocess_batch(batch: Dict[str, List]):  
    model_inputs = tokenizer(batch["text"], max_length=MAX_LENGTH, truncation=True, padding='max_length')    
    model_inputs["labels"] = copy.deepcopy(model_inputs['input_ids'])
    return model_inputs

_preprocessing_function = partial(_preprocess_batch)

# apply the preprocessing function to each batch in the dataset
encoded_small_dataset = small_dataset.map(
        _preprocessing_function,
        batched=True,
        remove_columns=["instruction", "response", "prompt", "answer"],
)
processed_dataset = encoded_small_dataset.filter(lambda rec: len(rec["input_ids"]) <= MAX_LENGTH)

# splitting dataset
split_dataset = processed_dataset.train_test_split(test_size=14, seed=0)
print(split_dataset)

# takes a list of samples from a Dataset and collate them into a batch, as a dictionary of PyTorch tensors.
data_collator = DataCollatorForSeq2Seq(
        model = model, tokenizer=tokenizer, max_length=MAX_LENGTH, pad_to_multiple_of=8, padding='max_length')
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()
from transformers import TrainingArguments, Trainer

EPOCHS = 3
LEARNING_RATE = 1e-4  
MODEL_SAVE_FOLDER_NAME = "qwen-model-0.6B"

# IMPORTANT for training
model.config.use_cache = False

training_args = TrainingArguments(
    output_dir=MODEL_SAVE_FOLDER_NAME,
    fp16=True,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    learning_rate=LEARNING_RATE,
    num_train_epochs=EPOCHS,
)

trainer = Trainer(
    model=model,
    processing_class=tokenizer,
    args=training_args,
    train_dataset=split_dataset['train'],
    eval_dataset=split_dataset["test"],
    data_collator=data_collator,
)

trainer.train()

# Save LoRA adapters (lightweight)
trainer.model.save_pretrained(MODEL_SAVE_FOLDER_NAME)






In [ ]:
# compare before and after using lora fine-tuning

In [49]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel


model_id = "Qwen/Qwen3-0.6B"

base_model = AutoModelForCausalLM.from_pretrained(model_id)
tokenizer = AutoTokenizer.from_pretrained(model_id)
base_model.resize_token_embeddings(len(tokenizer))

prompt = """### Instruction: Give best TTS and STT models in 2026.

### Response:
"""

inputs = tokenizer(prompt, return_tensors="pt")

output = base_model.generate(
    **inputs,
    max_new_tokens=100
)
# load LoRA adapter
lora_model = PeftModel.from_pretrained(base_model, "qwen-model-0.6B")


output_a = lora_model.generate(
    **inputs,
    max_new_tokens=500
)

print("=== BEFORE LoRA ===")
print(tokenizer.decode(output[0], skip_special_tokens=True))
print("="*100)
print("\n=== AFTER LoRA ===")
print(tokenizer.decode(output_a[0], skip_special_tokens=True))


Loading weights: 100%|██████████| 311/311 [00:00<00:00, 6043.13it/s]


=== BEFORE LoRA ===
### Instruction: Give best TTS and STT models in 2026.

### Response:
```
{
  "best_tts_models": [
    {
      "name": "Google TTS",
      "description": "A highly accurate and natural language generation model developed by Google, known for its ability to produce human-like speech.",
      "features": [
        "High-quality speech quality",
        "Natural speech patterns",
        "Supports multiple languages",
        "Can generate text in multiple languages",
        "Has a good understanding of complex sentences and grammar",
        "Is available in

=== AFTER LoRA ===
### Instruction: Give best TTS and STT models in 2026.

### Response:
The best TTS and STT models in 2026 can be:

TTS:
1. Azure Cognitive Text-to-Speech
2. Google Cloud TTS
3. Amazon Polly
4. Microsoft Azure TTS

STT:
1. Google Cloud Speech Recognition
2. Amazon Rekognition
3. Google Cloud NLP
4. Microsoft Azure NLP

Note that the specific models may vary depending on the platform and region 

In [52]:
print(sum( i.numel() for i in lora_model.parameters()))

print(sum( i.numel() for i in base_model.parameters()))

600364032
600364032


In [ ]:
# from peft import PeftModel

# # load base model again
# base_model = AutoModelForCausalLM.from_pretrained(model_id)


# base_model.resize_token_embeddings(len(tokenizer))

# # load LoRA adapter
# lora_model = PeftModel.from_pretrained(base_model, "qwen-model-0.6B")

# inputs = tokenizer(prompt, return_tensors="pt")

# output = lora_model.generate(
#     **inputs,
#     max_new_tokens=500
# )

# print("\n=== AFTER LoRA ===")
# print(tokenizer.decode(output[0], skip_special_tokens= True))

In [1]:
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

/home/shrutik/Srutik/12-Week-Internship/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model_id = "Qwen/Qwen3-0.6B"


In [7]:
from transformers import AutoTokenizer,AutoModelForCausalLM

In [3]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",   # normalfloat4 - store at 4bit
    bnb_4bit_compute_dtype="bfloat16", # compute at 16 bit
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config = bnb_config,
    device_map = "auto"
)

lora_configg = LoraConfig(
    r = 16,
    lora_alpha=32,
    lora_dropout=0.05,
    lora_bias=None,
    target_modules=["q_proj","v_proj"]
)

qlora_model = get_peft_model(model,lora_configg)

Loading weights: 100%|██████████| 311/311 [00:36<00:00,  8.56it/s]


In [8]:
tokenizer = AutoTokenizer.from_pretrained(model_id)

In [9]:
prompt = """### Instruction: Give best TTS and STT models in 2026.

### Response:
"""

inputs = tokenizer(prompt, return_tensors="pt")

In [10]:
output_q = model.generate(
    **inputs,
    max_new_tokens=200,
    do_sample=True,
    temperature=0.7,
    repetition_penalty=1.2,   
    no_repeat_ngram_size=3    
)

print("="*100)
print("\n=== AFTER QLoRA ===")
print(tokenizer.decode(output_q[0], skip_special_tokens=True))



=== AFTER QLoRA ===
### Instruction: Give best TTS and STT models in 2026.

### Response:
We can provide you with the latest available technologies for your needs. The top five are as follows:

1) **Pytorch-3D** – This is a major model that is expected to be more efficient than previous versions, especially in terms of computational efficiency.
2) **Swin-2B + Swin-T (VIT-B)** – These two pre-trained large vision languages will become very popular due to their high accuracy on both fine-tuned tasks and general performance across different domains.
3) **Mamba-XL - XLM-Raw** – A new architecture for language modeling that has been designed specifically for larger models.
4) **Hugging Face's GPT-9/17/38 etc.**, which have shown good results even after some updates over time.
5) **Rafaela’s Model** – With its specific design and training strategy, it might perform better than others.

These models should give you an idea of what we're going to


In [70]:
# find total parameters in model
print(sum( i.numel() for i in qlora_model.parameters()))

378142720


## RAG System
- importlibs
- data ingestion
- chunking
- embedding + vector store
- Query with LLM
- Citation support
- retrival
- reranking
- context builder
- LLM gen. answer

In [1]:
# import libraries

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceBgeEmbeddings
from langchain_community.vectorstores import FAISS
from sentence_transformers import cross_encoder
from transformers import pipeline

/home/shrutik/Srutik/12-Week-Internship/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# upload document

import tkinter as tk
from tkinter import filedialog

def browse_file():
    # Initializes a hidden Tkinter window
    root = tk.Tk()
    root.withdraw()
    
    # Opens a 'Browse' window and returns the selected file path
    file_path = filedialog.askopenfilename(
        title="Select a file",
        filetypes=[("Text files", "*.txt"), ("All files", "*.*")]
    )
    return file_path

selected_file = browse_file()

if selected_file:
    print(f"Selected file: {selected_file}")
    print("Document added successfully!")
else:
    print("No file was selected.")
loader = PyPDFLoader(selected_file)
document = loader.load()
# chunking

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 100
)
chunked_doc = splitter.split_documents(document)
# embedding + vector store

embedding = HuggingFaceBgeEmbeddings()

db = FAISS.from_documents(chunked_doc,embedding)
query = "what is embeddings?"
results = db.similarity_search(query,k = 3)
for i in results:
    print(i.page_content)

Selected file: /home/shrutik/Documents/doc.pdf
Document added successfully!


In [20]:
selected_file

'/home/shrutik/Documents/hrmsnotw.pdf'

In [2]:
%pip install tiktoken

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# straming response RAG

In [17]:
# ==============================
# 1. IMPORTS
# ==============================
import tkinter as tk
from tkinter import filedialog
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceBgeEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_community.llms import Ollama

from sentence_transformers import CrossEncoder


# ==============================
# 2. FILE UPLOAD
# ==============================
def browse_file():
    root = tk.Tk()
    root.withdraw()

    file_path = filedialog.askopenfilename(
        title="Select a PDF file",
        filetypes=[("PDF files", "*.pdf"), ("All files", "*.*")],
    )
    return file_path


selected_file = browse_file()

if not selected_file:
    print("No file selected")
    exit()
filename = os.path.basename(selected_file)
print(f"Loaded: {filename}")


# ==============================
# 3. LOAD DOCUMENT
# ==============================
loader = PyPDFLoader(selected_file)
documents = loader.load()


# ==============================
# 4. CHUNKING
# ==============================
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)

chunked_docs = splitter.split_documents(documents)


# ==============================
# 5. EMBEDDING + VECTOR STORE
# ==============================
embedding = HuggingFaceBgeEmbeddings(model_name="BAAI/bge-small-en")

db = FAISS.from_documents(chunked_docs, embedding)


# ==============================
# 6. RETRIEVER
# ==============================
retriever = db.as_retriever(search_type="mmr", search_kwargs={"k": 5})


# ==============================
# 7. LOAD FREE LLM (OLLAMA)
# ==============================
# Make sure you run:
# ollama run mistral

# llm = Ollama(model="phi3",options={
#     "num_predict": 512,
#     "num_ctx" : 1024
# })
llm = Ollama(
    model="phi3"        
)

# ==============================
# 8. RE-RANKER
# ==============================
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")



# ==============================
# 9. PROMPT TEMPLATE
# ==============================
prompt = PromptTemplate(
    template="""
You are a strict document QA system.

Rules:
1. Answer ONLY using the provided context
2. Do NOT use your own knowledge
3. If the answer is NOT explicitly in the context, say EXACTLY:
   "Not found in document"
4. Do NOT guess or infer the response

Context:
{context}

Question:
{question}

Answer:
""",
    input_variables=["context", "question"],
)


# ==============================
# 10. ASK QUERY LOOP
# ==============================
while True:

    
    query = input("\nAsk a question (or type 'exit' for end chat): ")
    MAX_TOKEN = 256
    from transformers import AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained("microsoft/phi-3-mini-4k-instruct")
    def count_token(query):
        return len(tokenizer.encode(query))

    token_count = count_token(query)

    if token_count > MAX_TOKEN:
        print(f"Input is too long,contain ({token_count} tokens),MAX TOKEN :{MAX_TOKEN}")


    if query.lower() == "exit":
        print("\n===========  **Chat Ended**  ===========\n")
        break

    print("\n================ QUESTION ================\n")
    print(query)

    # --------------------------
    # Retrieve documents
    # --------------------------
    docs = retriever.invoke(query)

    # --------------------------
    # Re-ranking
    # --------------------------
    pairs = [(query, doc.page_content) for doc in docs]
    scores = reranker.predict(pairs)

    reranked_docs = [doc for _, doc in sorted(zip(scores, docs), reverse=True)]

    # --------------------------
    # Context Builder
    # --------------------------
    context = "\n\n".join(
        [f"[Source {i+1}]: {doc.page_content}" for i, doc in enumerate(reranked_docs)]
    )

    # --------------------------
    # Final Prompt
    # --------------------------
    final_prompt = prompt.format(context=context, question=query)


    if token_count > MAX_TOKEN:
        print("prompt is too long,triming context...")

    # --------------------------
    # LLM RESPONSE
    # --------------------------
    if len(docs) == 0:
        print("Not found in document")
        continue

    print("\n================ ANSWER ================\n")

    # stream response
    response=""
    for chunk in llm.stream(final_prompt):
        response += chunk
        print(chunk, end="", flush=True)
    print(response)
    print("\n\n============  ************  ============\n")

# def logs():

Loaded: week 7-12 report.pdf


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4419.76it/s]
BertModel LOAD REPORT from: BAAI/bge-small-en
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 4885.16it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



===========  **Chat Ended**  ===========



In [ ]:
# RAG EVALUATION - RAGAS

In [2]:
%pip install ragas datasets langchain openai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 KB 2.5 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.2/178.2 KB 980.0 kB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 KB 390.2 kB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 3.7 MB/s eta 0:00:0000:0100:01m
Note: you may need to restart the kernel to use updated packages.


In [18]:
%pip install -U langchain-ollama

Note: you may need to restart the kernel to use updated packages.


In [1]:
data = {
    "question": [
        "What is RAG?",
        "What is LangChain?",
        "What is vector database?",
        "What is embedding?",
        "What is LLM?",
        "What is fine-tuning?",
        "What is prompt engineering?",
        "What is retrieval in RAG?",
        "What is hallucination in LLM?",
        "What is context window?"
    ],
    
    "answer": [
        "RAG combines retrieval and generation.",
        "LangChain is a framework for building LLM applications.",
        "A vector database stores embeddings for similarity search.",
        "Embedding is a numerical representation of text.",
        "LLM is a large language model trained on text data.",
        "Fine-tuning adapts a model to specific tasks.",
        "Prompt engineering is designing inputs for better outputs.",
        "Retrieval fetches relevant documents before generation.",
        "Hallucination is when LLM generates incorrect information.",
        "Context window is the amount of text an LLM can process."
    ],
    
    "reference": [
        "RAG Documentation",
        "LangChain Docs",
        "Vector DB Docs",
        "Embedding Guide",
        "LLM Guide",
        "Fine-tuning Guide",
        "Prompt Engineering Guide",
        "RAG Retrieval Docs",
        "LLM Safety Docs",
        "LLM Architecture Docs"
    ],
    
    "contexts": [
        ["RAG is a technique that retrieves documents and generates answers."],
        ["LangChain helps developers build applications using language models."],
        ["Vector databases store embeddings and allow similarity search."],
        ["Embeddings convert text into vectors for machine understanding."],
        ["LLMs are trained on large datasets for natural language tasks."],
        ["Fine-tuning modifies pre-trained models for specific use cases."],
        ["Prompt engineering improves model responses using better prompts."],
        ["Retrieval in RAG fetches relevant context before answering."],
        ["Hallucination occurs when models produce false information."],
        ["Context window defines how much input a model can handle."]
    ]
}

from datasets import Dataset
dataset = Dataset.from_dict(data)
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall
)


/home/shrutik/Srutik/12-Week-Internship/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/shrutik/Srutik/12-Week-Internship/.venv/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.12) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
/home/shrutik/Srutik/12-Week-Internship/.venv/lib/python3.10/site-packages/instructor/providers/gemini/client.py:5: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Pl

In [5]:
from langchain_ollama import ChatOllama
from langchain_ollama import OllamaEmbeddings

# LLM (Ollama)
llm = ChatOllama(model="phi3",timeout=120, 
    max_retries=3)

# Embeddings (Ollama)
embeddings = OllamaEmbeddings(model="nomic-embed-text")

from ragas import evaluate

result = evaluate(
    dataset,
    llm=llm,
    embeddings=embeddings,   
    metrics=[
        faithfulness,
        context_precision,
        answer_relevancy,
        context_recall
    ],
)

print(result)

Evaluating:   5%|▌         | 2/40 [02:39<45:20, 71.60s/it]   Exception raised in Job[0]: TimeoutError()
Exception raised in Job[1]: TimeoutError()
Exception raised in Job[2]: TimeoutError()
Exception raised in Job[3]: TimeoutError()
Exception raised in Job[4]: TimeoutError()
Exception raised in Job[5]: TimeoutError()
Exception raised in Job[6]: TimeoutError()
Exception raised in Job[8]: TimeoutError()
Exception raised in Job[9]: TimeoutError()
Exception raised in Job[10]: TimeoutError()
Exception raised in Job[11]: TimeoutError()
Exception raised in Job[12]: TimeoutError()
Exception raised in Job[13]: TimeoutError()
Exception raised in Job[14]: TimeoutError()
Evaluating:  42%|████▎     | 17/40 [05:07<04:57, 12.94s/it]Exception raised in Job[16]: TimeoutError()
Exception raised in Job[18]: TimeoutError()
Exception raised in Job[19]: TimeoutError()
Exception raised in Job[20]: TimeoutError()
Exception raised in Job[21]: TimeoutError()
Exception raised in Job[22]: TimeoutError()
Exception

{'faithfulness': nan, 'context_precision': 0.0000, 'answer_relevancy': nan, 'context_recall': 0.2500}
